# Athar + ACL-OCL Paper-Matched Non-Citation Pipeline

In [ ]:
import os
# Set WANDB_API_KEY as an environment variable (never commit an API key).

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ============================================================================

# 1. Εγκατάσταση βιβλιοθηκών

# ============================================================================

In [ ]:
# Notebook dependency command:
!pip -q install -U huggingface_hub hf_xet pyarrow rapidfuzz pysbd tqdm

# ============================================================================

# 2. Imports και configuration

# ============================================================================


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pysbd

from rapidfuzz import fuzz, process
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download

warnings.filterwarnings("ignore")

ATHAR_CSV = Path("/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/citation_sentiment_corpus.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/athar_acl_ocl_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

ACL_OCL_REPO = "WINGNUS/ACL-OCL"
ACL_OCL_REVISION = "300ee6c5e5629d042bfc07cbc406e2f330b53659"
ACL_OCL_PARQUET = "acl-publication-info.74k.v2.parquet"

MIN_WORDS = 6
MAX_WORDS = 100
MIN_CHARS = 35
MAX_CHARS = 900

ATHAR_FUZZY_THRESHOLD = 90
MAX_EXCLUDED_EXAMPLES_PER_REASON = 200

print("Output directory:", OUTPUT_DIR)

# ============================================================================

# 3. Upload του Athar CSV αν χρειάζεται

# ============================================================================

In [ ]:
if not ATHAR_CSV.exists():
    try:
        from google.colab import files

        print("Upload the Athar CSV file.")
        uploaded = files.upload()

        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file uploaded.")

        preferred = "citation_sentiment_corpus.csv"
        selected = preferred if preferred in csv_files else csv_files[0]
        ATHAR_CSV = Path("/content") / selected

    except ImportError:
        raise FileNotFoundError(
            f"{ATHAR_CSV} not found. Set ATHAR_CSV to the correct path."
        )

print("Athar CSV:", ATHAR_CSV)
assert ATHAR_CSV.exists()

# ============================================================================

# 4. Φόρτωση του Athar και polarity mapping

# ============================================================================

In [ ]:
ATHAR_COLUMNS = [
    "Source_Paper_ID",
    "Target_Paper_ID",
    "Sentiment",
    "Citation_Text",
]

athar = pd.read_csv(
    ATHAR_CSV,
    header=None,
    names=ATHAR_COLUMNS,
    dtype=str,
    keep_default_na=False,
)

for col in ATHAR_COLUMNS:
    athar[col] = athar[col].astype(str).str.strip()

POLARITY_MAP = {
    "n": 0,
    "o": 1,
    "p": 2,
}

assert set(athar["Sentiment"].unique()).issubset(POLARITY_MAP.keys())

athar["polarity"] = athar["Sentiment"].map(POLARITY_MAP).astype("int8")
athar["is_citation"] = np.int8(1)

print("Rows:", len(athar))
print("Unique source papers:", athar["Source_Paper_ID"].nunique())
print("Unique target papers:", athar["Target_Paper_ID"].nunique())
print()
print("Polarity distribution:")
print(athar["polarity"].value_counts().sort_index())

display(athar.head())

# ============================================================================

# 5. Duplicate diagnostics χωρίς αφαίρεση Athar rows

# ============================================================================

In [ ]:
duplicate_diagnostics = (
    athar.groupby(
        ["Source_Paper_ID", "Citation_Text"],
        dropna=False,
    )
    .agg(
        row_count=("Target_Paper_ID", "size"),
        unique_targets=("Target_Paper_ID", "nunique"),
        unique_polarities=("polarity", "nunique"),
        targets=(
            "Target_Paper_ID",
            lambda x: " | ".join(sorted(set(x)))
        ),
        polarities=(
            "polarity",
            lambda x: " | ".join(
                map(str, sorted(set(x)))
            )
        ),
    )
    .reset_index()
)

duplicate_diagnostics = duplicate_diagnostics[
    duplicate_diagnostics["row_count"] > 1
].copy()

duplicate_diagnostics.to_csv(
    OUTPUT_DIR / "athar_duplicate_diagnostics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Exact duplicated full rows:", int(athar.duplicated().sum()))
print("Repeated Source_Paper_ID + Citation_Text groups:", len(duplicate_diagnostics))
display(duplicate_diagnostics.head(10))

# ============================================================================

# 6. Λήψη μόνο του ACL-OCL parquet

# ============================================================================

In [ ]:
def locate_acl_ocl_parquet():
    api = HfApi()

    repo_files = api.list_repo_files(
        repo_id=ACL_OCL_REPO,
        repo_type="dataset",
        revision=ACL_OCL_REVISION,
    )

    if ACL_OCL_PARQUET in repo_files:
        return ACL_OCL_PARQUET

    candidates = [
        name
        for name in repo_files
        if name.lower().endswith(".parquet")
        and "publication-info" in name.lower()
    ]

    if not candidates:
        raise FileNotFoundError(
            "No ACL-OCL publication-info parquet file found."
        )

    v2_candidates = [
        name
        for name in candidates
        if ".v2." in name.lower()
    ]

    return sorted(v2_candidates or candidates)[0]


acl_parquet_name = locate_acl_ocl_parquet()

acl_parquet_path = Path(
    hf_hub_download(
        repo_id=ACL_OCL_REPO,
        filename=acl_parquet_name,
        repo_type="dataset",
        revision=ACL_OCL_REVISION,
    )
)

print("ACL-OCL parquet:", acl_parquet_name)
print("Local path:", acl_parquet_path)
print(
    "Size (MB):",
    round(acl_parquet_path.stat().st_size / 1024**2, 2),
)


# ============================================================================

# 7. SHA-256 fingerprint

# ============================================================================

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            hasher.update(chunk)

    return hasher.hexdigest()


ACL_OCL_SHA256 = sha256_file(acl_parquet_path)

print("SHA-256:")
print(ACL_OCL_SHA256)

# ============================================================================

# 8. Matching των Athar Source_Paper_ID με ACL-OCL acl_id

# ============================================================================

In [ ]:
wanted_ids = set(
    athar["Source_Paper_ID"].unique()
)

parquet_file = pq.ParquetFile(
    acl_parquet_path
)

required_columns = {
    "acl_id",
    "full_text",
}

missing_columns = (
    required_columns
    - set(parquet_file.schema.names)
)

if missing_columns:
    raise ValueError(
        f"Missing required ACL-OCL columns: {missing_columns}"
    )

matched_chunks = []

for batch in tqdm(
    parquet_file.iter_batches(
        batch_size=2500,
        columns=["acl_id", "full_text"],
    ),
    desc="Matching Athar source papers",
):
    part = batch.to_pandas()

    part["acl_id"] = (
        part["acl_id"]
        .astype(str)
        .str.strip()
    )

    matched = part[
        part["acl_id"].isin(wanted_ids)
    ].copy()

    if not matched.empty:
        matched_chunks.append(matched)

if matched_chunks:
    acl_subset = pd.concat(
        matched_chunks,
        ignore_index=True,
    )
else:
    acl_subset = pd.DataFrame(
        columns=["acl_id", "full_text"]
    )

acl_subset["full_text"] = (
    acl_subset["full_text"]
    .fillna("")
    .astype(str)
)

acl_subset["_text_length"] = (
    acl_subset["full_text"]
    .str.len()
)

acl_subset = (
    acl_subset
    .sort_values(
        ["acl_id", "_text_length"],
        ascending=[True, False],
    )
    .drop_duplicates(
        "acl_id",
        keep="first",
    )
    .drop(
        columns="_text_length"
    )
    .reset_index(drop=True)
)

found_ids = set(
    acl_subset["acl_id"]
)

missing_ids = sorted(
    wanted_ids - found_ids
)

pd.DataFrame(
    {"Source_Paper_ID": missing_ids}
).to_csv(
    OUTPUT_DIR / "unmatched_source_papers.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Athar source papers:", len(wanted_ids))
print("Matched ACL-OCL papers:", len(found_ids))
print("Unmatched papers:", len(missing_ids))
print(
    "Matched papers with full text:",
    int(
        acl_subset["full_text"]
        .str.strip()
        .ne("")
        .sum()
    ),
)

# ============================================================================

# 9. Sentence segmentation και filtering functions

# ============================================================================

In [ ]:
segmenter = pysbd.Segmenter(
    language="en",
    clean=False,
)

WS_RE = re.compile(r"\s+")
URL_RE = re.compile(
    r"(https?://|www\.)",
    re.I,
)
EMAIL_RE = re.compile(
    r"\b[\w.\-+]+@[\w.\-]+\.\w+\b",
    re.I,
)

NUMERIC_CIT_RE = re.compile(
    r"\["
    r"\s*\d{1,3}[a-z]?"
    r"(?:\s*[-–—]\s*\d{1,3}[a-z]?)?"
    r"(?:"
    r"\s*[,;]\s*"
    r"\d{1,3}[a-z]?"
    r"(?:\s*[-–—]\s*\d{1,3}[a-z]?)?"
    r")*"
    r"\s*\]"
)

PLACEHOLDER_CIT_RE = re.compile(
    r"\[(?:CIT|CITE|REF|BIBR)(?:ATION)?[^\]]*\]",
    re.I,
)

NARRATIVE_AUTHOR_YEAR_RE = re.compile(
    r"\b"
    r"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]+"
    r"(?:\s+et\s+al\.)?"
    r"(?:"
    r"\s+(?:and|&)\s+"
    r"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]+"
    r")?"
    r"(?:['’]s)?"
    r"\s*\("
    r"\s*(?:19|20)\d{2}[a-z]?"
    r"\s*\)"
)

PAREN_AUTHOR_YEAR_RE = re.compile(
    r"\("
    r"(?=[^()]{0,220}(?:19|20)\d{2}[a-z]?)"
    r"(?=[^()]{0,220}\b[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{2,}\b)"
    r"[^()]{0,220}"
    r"\)"
)

CAPTION_RE = re.compile(
    r"^\s*(?:figure|fig\.?|table)"
    r"\s+\d+[A-Za-z]?"
    r"\s*[:.]\s*",
    re.I,
)

SECTION_HEADING_RE = re.compile(
    r"^\s*\d+(?:\.\d+)*"
    r"\s+[A-Z][^\n]{0,100}$"
)

REFERENCE_ENTRY_RE = re.compile(
    r"^\s*"
    r"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]+,"
    r"\s*(?:[A-Z]\.?\s*)+"
    r".{0,180}"
    r"\b(?:19|20)\d{2}[a-z]?\b"
)

REF_HEADING_RE = re.compile(
    r"(?im)"
    r"(?:^|\n)\s*"
    r"(?:references|bibliography|works\s+cited|literature\s+cited)"
    r"\s*(?:\n|$)"
)


def normalize_unicode(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    text = text.replace(
        "\u00ad",
        "",
    )

    text = text.replace(
        "\u200b",
        "",
    )

    return text


def normalize_whitespace(text):
    return WS_RE.sub(
        " ",
        normalize_unicode(text),
    ).strip()


def normalize_for_match(text):
    text = (
        normalize_whitespace(text)
        .lower()
    )

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE,
    )

    return WS_RE.sub(
        " ",
        text,
    ).strip()


def strip_reference_section(full_text):
    text = normalize_unicode(
        full_text
    )

    if not text.strip():
        return ""

    headings = list(
        REF_HEADING_RE.finditer(text)
    )

    if not headings:
        return text

    late_headings = [
        match
        for match in headings
        if match.start() >= 0.45 * len(text)
    ]

    if not late_headings:
        return text

    return text[
        :late_headings[0].start()
    ].rstrip()


def has_citation_pattern(sentence):
    text = normalize_whitespace(
        sentence
    )

    if NUMERIC_CIT_RE.search(text):
        return True, "numeric_bracket_citation"

    if PLACEHOLDER_CIT_RE.search(text):
        return True, "citation_placeholder"

    if NARRATIVE_AUTHOR_YEAR_RE.search(text):
        return True, "narrative_author_year"

    for match in PAREN_AUTHOR_YEAR_RE.finditer(text):
        fragment = match.group(0)

        strong_pattern = (
            re.search(
                r"\b(?:et\s+al\.|and|&|see|cf\.)\b",
                fragment,
                re.I,
            )
            or re.search(
                r"\b[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{2,}"
                r"\s*,\s*(?:19|20)\d{2}",
                fragment,
            )
            or re.search(
                r"\b[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{2,}"
                r"\s+et\s+al\.",
                fragment,
                re.I,
            )
        )

        if strong_pattern:
            return True, "parenthetical_author_year"

    return False, None


def basic_quality_rejection(sentence):
    text = normalize_whitespace(
        sentence
    )

    if not text:
        return "empty"

    char_count = len(text)

    words = re.findall(
        r"\b[\w'’\-]+\b",
        text,
        flags=re.UNICODE,
    )

    word_count = len(words)

    if char_count < MIN_CHARS:
        return "too_short_chars"

    if char_count > MAX_CHARS:
        return "too_long_chars"

    if word_count < MIN_WORDS:
        return "too_few_words"

    if word_count > MAX_WORDS:
        return "too_many_words"

    if URL_RE.search(text):
        return "url"

    if EMAIL_RE.search(text):
        return "email"

    if CAPTION_RE.search(text):
        return "figure_table_caption"

    if (
        SECTION_HEADING_RE.match(text)
        and word_count <= 14
        and not re.search(r"[.!?]$", text)
    ):
        return "section_heading"

    if REFERENCE_ENTRY_RE.match(text):
        return "reference_like_entry"

    nonspace_chars = [
        char
        for char in text
        if not char.isspace()
    ]

    if nonspace_chars:
        alpha_ratio = (
            sum(
                char.isalpha()
                for char in nonspace_chars
            )
            / len(nonspace_chars)
        )

        if alpha_ratio < 0.45:
            return "low_alpha_ratio"

    math_symbols = len(
        re.findall(
            r"[=<>±×÷∑∏√∞{}|]",
            text,
        )
    )

    if (
        math_symbols >= 5
        and math_symbols / max(1, char_count) > 0.02
    ):
        return "equation_or_math_fragment"

    has_citation, reason = (
        has_citation_pattern(text)
    )

    if has_citation:
        return reason

    return None


def athar_similarity_rejection(
    sentence,
    known_contexts,
):
    if not known_contexts:
        return False, 0.0

    normalized_sentence = normalize_for_match(
        sentence
    )

    if len(normalized_sentence) < 20:
        return False, 0.0

    for known in known_contexts:
        if not known:
            continue

        if normalized_sentence == known:
            return True, 100.0

        if (
            min(
                len(normalized_sentence),
                len(known),
            ) >= 40
            and (
                normalized_sentence in known
                or known in normalized_sentence
            )
        ):
            return True, 100.0

    match = process.extractOne(
        normalized_sentence,
        known_contexts,
        scorer=fuzz.WRatio,
        score_cutoff=ATHAR_FUZZY_THRESHOLD,
    )

    if match is not None:
        return True, float(match[1])

    return False, 0.0


print("Filtering functions ready.")

# ============================================================================

# 10. Εξαγωγή clean non-citation candidates

# ============================================================================

In [ ]:
athar_known_by_paper = {
    paper_id: list(
        dict.fromkeys(
            normalize_for_match(text)
            for text in group["Citation_Text"].tolist()
            if normalize_for_match(text)
        )
    )
    for paper_id, group
    in athar.groupby(
        "Source_Paper_ID",
        sort=False,
    )
}

athar_count_by_paper = (
    athar
    .groupby("Source_Paper_ID")
    .size()
    .to_dict()
)

acl_map = dict(
    zip(
        acl_subset["acl_id"],
        acl_subset["full_text"],
    )
)

candidate_records = []
excluded_counts = Counter()
excluded_examples = defaultdict(list)
paper_stats = {}

for paper_id in tqdm(
    sorted(wanted_ids),
    desc="Extracting non-citation candidates",
):

    full_text = str(
        acl_map.get(
            paper_id,
            "",
        )
        or ""
    )

    stats = {
        "Source_Paper_ID": paper_id,
        "athar_citation_instances": int(
            athar_count_by_paper.get(
                paper_id,
                0,
            )
        ),
        "matched_in_acl_ocl": int(
            paper_id in acl_map
        ),
        "has_nonempty_full_text": int(
            bool(full_text.strip())
        ),
        "raw_segmented_sentences": 0,
        "clean_non_citation_candidates": 0,
    }

    if not full_text.strip():
        paper_stats[paper_id] = stats
        continue

    body_text = strip_reference_section(
        full_text
    )

    try:
        raw_sentences = segmenter.segment(
            body_text
        )
    except Exception:
        raw_sentences = re.split(
            r"(?<=[.!?])\s+(?=[A-Z])",
            body_text,
        )

    stats["raw_segmented_sentences"] = len(
        raw_sentences
    )

    known_contexts = athar_known_by_paper.get(
        paper_id,
        [],
    )

    seen_in_paper = set()

    for sentence_index, raw_sentence in enumerate(
        raw_sentences
    ):
        sentence = normalize_whitespace(
            raw_sentence
        )

        normalized_sentence = normalize_for_match(
            sentence
        )

        fuzzy_score = np.nan

        if not normalized_sentence:
            reason = "empty"

        elif normalized_sentence in seen_in_paper:
            reason = "duplicate_within_paper"

        else:
            reason = basic_quality_rejection(
                sentence
            )

            if reason is None:
                reject, fuzzy_score = (
                    athar_similarity_rejection(
                        sentence,
                        known_contexts,
                    )
                )

                if reject:
                    reason = "athar_context_similarity"

        if reason is not None:
            excluded_counts[reason] += 1

            if (
                len(
                    excluded_examples[reason]
                )
                < MAX_EXCLUDED_EXAMPLES_PER_REASON
            ):
                excluded_examples[reason].append(
                    {
                        "Source_Paper_ID": paper_id,
                        "sentence_index": sentence_index,
                        "sentence": sentence,
                        "exclusion_reason": reason,
                        "athar_fuzzy_score": fuzzy_score,
                    }
                )

            continue

        seen_in_paper.add(
            normalized_sentence
        )

        candidate_records.append(
            {
                "Source_Paper_ID": paper_id,
                "sentence_index": sentence_index,
                "sentence": sentence,
                "is_citation": np.int8(0),
                "polarity": np.int8(1),
                "source": "ACL-OCL",
                "sampling_status": "candidate",
            }
        )

        stats["clean_non_citation_candidates"] += 1

    paper_stats[paper_id] = stats


candidates = pd.DataFrame(
    candidate_records
)

excluded_report = pd.DataFrame(
    [
        {
            "exclusion_reason": reason,
            "count": count,
        }
        for reason, count
        in excluded_counts.most_common()
    ]
)

excluded_sample = pd.DataFrame(
    [
        row
        for rows
        in excluded_examples.values()
        for row
        in rows
    ]
)

candidates.to_csv(
    OUTPUT_DIR / "acl_non_citation_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)

excluded_report.to_csv(
    OUTPUT_DIR / "excluded_sentences_report.csv",
    index=False,
    encoding="utf-8-sig",
)

excluded_sample.to_csv(
    OUTPUT_DIR / "excluded_sentences_sample.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Clean non-citation candidates:",
    len(candidates),
)

display(excluded_report)
display(candidates.head())

# ============================================================================

# 11. Paper-matched sampling

# ============================================================================

In [ ]:
if not candidates.empty:
    candidate_groups = {
        paper_id: group.copy()
        for paper_id, group
        in candidates.groupby(
            "Source_Paper_ID",
            sort=False,
        )
    }
else:
    candidate_groups = {}

selected_parts = []
report_rows = []

for paper_id in sorted(wanted_ids):

    target_count = int(
        athar_count_by_paper.get(
            paper_id,
            0,
        )
    )

    pool = candidate_groups.get(
        paper_id,
        pd.DataFrame(
            columns=candidates.columns
        ),
    )

    available_count = len(pool)

    sample_count = min(
        target_count,
        available_count,
    )

    if sample_count > 0:
        seed_text = (
            f"{RANDOM_SEED}|{paper_id}"
        )

        paper_seed = int(
            hashlib.sha256(
                seed_text.encode("utf-8")
            ).hexdigest()[:8],
            16,
        )

        selected = pool.sample(
            n=sample_count,
            replace=False,
            random_state=paper_seed,
        ).copy()

        selected["sampling_status"] = (
            "paper_matched_exact"
            if sample_count == target_count
            else "paper_matched_partial"
        )

        selected_parts.append(
            selected
        )

    base_stats = paper_stats.get(
        paper_id,
        {
            "Source_Paper_ID": paper_id,
            "athar_citation_instances": target_count,
            "matched_in_acl_ocl": 0,
            "has_nonempty_full_text": 0,
            "raw_segmented_sentences": 0,
            "clean_non_citation_candidates": 0,
        },
    )

    report_rows.append(
        {
            **base_stats,
            "sampled_non_citation": sample_count,
            "shortfall": (
                target_count
                - sample_count
            ),
            "exact_1_to_1_for_paper": int(
                sample_count == target_count
            ),
        }
    )


if selected_parts:
    selected_non_citation = pd.concat(
        selected_parts,
        ignore_index=True,
    )
else:
    selected_non_citation = pd.DataFrame(
        columns=candidates.columns
    )

paper_report = pd.DataFrame(
    report_rows
)

selected_non_citation.to_csv(
    OUTPUT_DIR / "athar_matched_non_citation_sentences.csv",
    index=False,
    encoding="utf-8-sig",
)

paper_report.to_csv(
    OUTPUT_DIR / "paper_matching_report.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Athar citation rows:", len(athar))
print(
    "Selected non-citation rows:",
    len(selected_non_citation),
)
print(
    "Total shortfall:",
    int(
        paper_report["shortfall"].sum()
    ),
)
print(
    "Papers with exact 1:1 matching:",
    int(
        paper_report[
            "exact_1_to_1_for_paper"
        ].sum()
    ),
    "/",
    len(paper_report),
)

display(
    paper_report.sort_values(
        [
            "shortfall",
            "athar_citation_instances",
        ],
        ascending=[
            False,
            False,
        ],
    ).head(20)
)


# ============================================================================

# 12. Athar citation rows για το τελικό schema

# ============================================================================

In [ ]:
athar_citations_out = pd.DataFrame(
    {
        "Source_Paper_ID": athar["Source_Paper_ID"],
        "Target_Paper_ID": athar["Target_Paper_ID"],
        "sentence": athar["Citation_Text"],
        "is_citation": np.int8(1),
        "polarity": athar["polarity"].astype("int8"),
        "source": "Athar",
        "sampling_status": "original",
    }
)

athar_citations_out.to_csv(
    OUTPUT_DIR / "athar_citation_sentences.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    athar_citations_out.head()
)

# ============================================================================

# 13. Δημιουργία τελικού CSV

# ============================================================================

In [ ]:
FINAL_COLUMNS = [
    "Source_Paper_ID",
    "sentence",
    "is_citation",
    "polarity",
    "source",
    "sampling_status",
]

athar_final = (
    athar_citations_out[
        FINAL_COLUMNS
    ]
    .copy()
)

non_citation_final = (
    selected_non_citation[
        FINAL_COLUMNS
    ]
    .copy()
)

non_citation_final["is_citation"] = np.int8(0)
non_citation_final["polarity"] = np.int8(1)

final_df = pd.concat(
    [
        athar_final,
        non_citation_final,
    ],
    ignore_index=True,
)

final_df = (
    final_df
    .sample(
        frac=1.0,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

final_df["is_citation"] = (
    final_df["is_citation"]
    .astype("int8")
)

final_df["polarity"] = (
    final_df["polarity"]
    .astype("int8")
)

FINAL_CSV = (
    OUTPUT_DIR
    / "final_citation_non_citation_dataset.csv"
)

final_df.to_csv(
    FINAL_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", FINAL_CSV)
print("Rows:", len(final_df))
print("Columns:", list(final_df.columns))

display(
    final_df.head(10)
)

# ============================================================================

# 14. Validation του τελικού dataset

# ============================================================================

In [ ]:
EXPECTED_FINAL_COLUMNS = [
    "Source_Paper_ID",
    "sentence",
    "is_citation",
    "polarity",
    "source",
    "sampling_status",
]

assert (
    list(final_df.columns)
    == EXPECTED_FINAL_COLUMNS
)

assert not final_df[
    EXPECTED_FINAL_COLUMNS
].isna().any().any()

assert set(
    final_df["is_citation"].unique()
).issubset({0, 1})

assert set(
    final_df["polarity"].unique()
).issubset({0, 1, 2})

assert (
    final_df.loc[
        final_df["is_citation"] == 0,
        "polarity",
    ]
    == 1
).all()

assert (
    final_df.loc[
        final_df["source"] == "Athar",
        "is_citation",
    ]
    == 1
).all()

athar_normalized_pairs = set(
    zip(
        athar["Source_Paper_ID"],
        athar["Citation_Text"].map(
            normalize_for_match
        ),
    )
)

exact_overlap = []

for row in non_citation_final.itertuples(
    index=False
):
    key = (
        row.Source_Paper_ID,
        normalize_for_match(
            row.sentence
        ),
    )

    if key in athar_normalized_pairs:
        exact_overlap.append(
            key
        )

assert not exact_overlap

print("VALIDATION PASSED")
print()
print(
    "Final columns:",
    list(final_df.columns),
)
print(
    "Citation rows:",
    int(
        (
            final_df["is_citation"] == 1
        ).sum()
    ),
)
print(
    "Non-citation rows:",
    int(
        (
            final_df["is_citation"] == 0
        ).sum()
    ),
)

print()
print("Polarity distribution:")
print(
    final_df[
        "polarity"
    ]
    .value_counts()
    .sort_index()
)

print()
print("is_citation x polarity:")
display(
    pd.crosstab(
        final_df["is_citation"],
        final_df["polarity"],
        margins=True,
    )
)

# ============================================================================

# 15. Pipeline summary

# ============================================================================

In [ ]:
summary = {
    "random_seed": RANDOM_SEED,
    "athar_rows": int(len(athar)),
    "athar_unique_source_papers": int(
        athar[
            "Source_Paper_ID"
        ].nunique()
    ),
    "acl_ocl_repo": ACL_OCL_REPO,
    "acl_ocl_revision": ACL_OCL_REVISION,
    "acl_ocl_parquet": acl_parquet_name,
    "acl_ocl_parquet_sha256": ACL_OCL_SHA256,
    "matched_source_papers": int(
        len(found_ids)
    ),
    "unmatched_source_papers": int(
        len(missing_ids)
    ),
    "clean_non_citation_candidates": int(
        len(candidates)
    ),
    "selected_non_citation_rows": int(
        len(selected_non_citation)
    ),
    "total_non_citation_shortfall": int(
        paper_report["shortfall"].sum()
    ),
    "final_rows": int(
        len(final_df)
    ),
    "final_citation_rows": int(
        (
            final_df["is_citation"] == 1
        ).sum()
    ),
    "final_non_citation_rows": int(
        (
            final_df["is_citation"] == 0
        ).sum()
    ),
    "polarity_mapping": {
        "negative": 0,
        "neutral_objective": 1,
        "positive": 2,
        "non_citation": 1,
    },
}

with open(
    OUTPUT_DIR / "pipeline_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

summary_df = pd.DataFrame(
    [
        {
            "metric": key,
            "value": (
                json.dumps(
                    value,
                    ensure_ascii=False,
                )
                if isinstance(
                    value,
                    dict,
                )
                else value
            ),
        }
        for key, value
        in summary.items()
    ]
)

summary_df.to_csv(
    OUTPUT_DIR / "pipeline_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(summary_df)

# ============================================================================

# 16. Manual validation sample

# ============================================================================

In [ ]:
sample_size = min(
    500,
    len(selected_non_citation),
)

if sample_size > 0:

    manual_sample = (
        selected_non_citation
        .sample(
            n=sample_size,
            random_state=RANDOM_SEED,
        )[
            [
                "Source_Paper_ID",
                "sentence",
            ]
        ]
        .copy()
    )

    manual_sample[
        "expected_is_citation"
    ] = 0

    manual_sample[
        "manual_is_citation"
    ] = ""

    manual_sample[
        "manual_notes"
    ] = ""

    manual_sample.to_csv(
        OUTPUT_DIR / "manual_validation_non_citation_sample.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "Saved manual validation sample."
    )

else:
    print(
        "No non-citation rows available."
    )

# ============================================================================

# Παραγόμενα αρχεία

# ============================================================================


# ============================================================================

# 17. Προαιρετικό ZIP των outputs

# ============================================================================

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/athar_acl_ocl_outputs",
    "zip",
    root_dir=OUTPUT_DIR,
)

print("ZIP:", zip_path)

try:
    from google.colab import files

    # Uncomment if you want automatic download:
    # files.download(zip_path)

except ImportError:
    pass